In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/bbc_dataset.csv")
print(f"Loaded {len(df)} articles.")
df.head()


Loaded 2225 articles.


,category,article,summary,sentences
0,entertainment,Musicians to tackle US red tape Musicians' gro...,Nigel McCune from the Musicians' Union said Br...,"[""Musicians to tackle US red tape Musicians' g..."
1,entertainment,"U2's desire to be number one U2, who have won ...",But they still want more.They have to want to ...,"[""U2's desire to be number one U2, who have wo..."
2,entertainment,Rocker Doherty in on-stage fight Rock singer P...,"Babyshambles, which he formed after his acrimo...","[""Rocker Doherty in on-stage fight Rock singer..."
3,entertainment,Snicket tops US box office chart The film adap...,A Series of Unfortunate Events also stars Scot...,"[""Snicket tops US box office chart The film ad..."
4,entertainment,Ocean's Twelve raids box office Ocean's Twelve...,"Ocean's Twelve, the crime caper sequel starrin...","[""Ocean's Twelve raids box office Ocean's Twel..."


In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize
import numpy as np
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

# Load a small, fast Sentence-BERT model
model = SentenceTransformer("all-MiniLM-L6-v2")


/Users/mac/bert_summarizer/paper_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to /Users/mac/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mac/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
pip install numpy==1.26.4 --force-reinstall


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 1.1 MB/s  0:00:17m0:00:0100:01m
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.4
    Uninstalling numpy-2.3.4:
      Successfully uninstalled numpy-2.3.4
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install --upgrade torch sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [5]:
def summarize_text(text, compression_ratio=0.3):
    """
    Summarize a given text using BERT embeddings + cosine similarity.
    """
    # Split into sentences
    sentences = sent_tokenize(text)
    if len(sentences) < 3:
        return text  # skip too-short texts

    # Generate embeddings for all sentences
    embeddings = model.encode(sentences)

    # Compute pairwise similarity between sentences
    sim_matrix = cosine_similarity(embeddings)

    # Sentence importance score = sum of similarities
    scores = sim_matrix.sum(axis=1)

    # Select top N sentences
    n_sentences = max(1, int(len(sentences) * compression_ratio))
    top_idx = np.argsort(scores)[-n_sentences:]
    top_idx.sort()

    # Combine selected sentences in original order
    summary = " ".join([sentences[i] for i in top_idx])
    return summary


In [1]:
import torch, numpy
print("Torch:", torch.__version__)
print("NumPy:", numpy.__version__)

x = torch.tensor([1.0, 2.0, 3.0])
print("Tensor → NumPy:", x.numpy())  # should work with no RuntimeError


Torch: 2.2.2
NumPy: 1.26.4
Tensor → NumPy: [1. 2. 3.]


In [8]:

sample_text = df["article"].iloc[0]
reference_summary = df["summary"].iloc[0]
generated_summary = summarize_text(sample_text)
print("Generated Summary:\n", generated_summary)
print("Reference Summary:\n", reference_summary)


Generated Summary:
 Musicians to tackle US red tape Musicians' groups are to tackle US visa regulations which are blamed for hindering British acts' chances of succeeding across the Atlantic. Nigel McCune from the Musicians' Union said British musicians are "disadvantaged" compared to their US counterparts. "The US is the world's biggest music market, which means something has to be done about the creaky bureaucracy," says Mr McCune. The Musicians' Union stance is being endorsed by the Music Managers' Forum (MMF), who say British artists face "an uphill struggle" to succeed in the US, thanks to the tough visa requirements, which are also seen as impractical. A Department for Media, Culture and Sport spokeswoman said: "We're aware that people are experiencing problems, and are working with the US embassy and record industry to see what we can do about it."
Reference Summary:
 Nigel McCune from the Musicians' Union said British musicians are "disadvantaged" compared to their US counterpa

In [9]:
from rouge import Rouge
rouge = Rouge()

scores = rouge.get_scores(generated_summary, reference_summary)
print(scores)


[{'rouge-1': {'r': 0.7027027027027027, 'p': 0.7722772277227723, 'f': 0.7358490516148986}, 'rouge-2': {'r': 0.6103896103896104, 'p': 0.706766917293233, 'f': 0.6550522598351323}, 'rouge-l': {'r': 0.7027027027027027, 'p': 0.7722772277227723, 'f': 0.7358490516148986}}]


In [10]:
sampled_df = df.sample(10, random_state=42)
sampled_df["generated_summary"] = sampled_df["article"].apply(summarize_text)

# Evaluate average ROUGE
rouge_scores = []
for i, row in sampled_df.iterrows():
    s = rouge.get_scores(row["generated_summary"], row["summary"])
    rouge_scores.append(s[0]["rouge-1"]["f"])

print(f"Average ROUGE-1 F1: {np.mean(rouge_scores):.3f}")


Average ROUGE-1 F1: 0.629
